In [ ]:
# =============================================================================
# PYBAMM EIS SIMULATION: CG50 CATHODE vs Li METAL HALF-CELL
#
# Corrected version:
#   1. Controls cathode SOC directly via initial positive electrode concentration
#   2. Uses wider frequency range to capture diffusion tail
#   3. Keeps SEI enabled so film-resistance/SEI effects are retained
#   4. Starts with SPM; change to SPMe only after debugging
#   5. Adds high-frequency-intercept-corrected Nyquist plot
#   6. Writes the CSV when save_results=True
#
# Convention:
#   Cathode/NMC working electrode = POSITIVE branch
#   Li metal counter/reference     = NEGATIVE branch
#
# Therefore:
#   options["working electrode"] = "positive"
# =============================================================================

import csv
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pybamm

# =============================================================================
# USER SETTINGS
# =============================================================================

# Start with SPM for speed and stability.
# After this works, try "SPMe".
# Use "DFN" only after SPM/SPMe are working.
# MODEL_NAME = "SPMe"        # options: "SPM", "SPMe", "DFN"
MODEL_NAME = "SPMe"
# MODEL_NAME = "DFN"

# Nominal SOC values to simulate.
# For a half-cell, these are treated as cathode stoichiometry-like values
# unless POSITIVE_STO_MAPPING = "one_minus".
# soc_list = [0.2, 0.5, 0.8]
soc_list = [0.2,0.5,0.7]
# Mapping from user SOC to positive-electrode stoichiometry.
# Use:
#   "direct"    -> sto_p = SOC
#   "one_minus" -> sto_p = 1 - SOC
#
# For cathode vs Li half-cell qOCV data, "direct" is often the best starting point
# if your qOCV sto axis was constructed as a cathode SOC-like coordinate.
POSITIVE_STO_MAPPING = "direct"
# POSITIVE_STO_MAPPING = "one_minus"

# Wider frequency range to capture diffusion tail.
# Starts at 1e-2 Hz so the visible real-axis return is the low-frequency end.
frequencies = np.logspace(-2, 6, 70)

# Geometry
USE_EL_CELL_DISC_GEOMETRY = True
disc_diameter_mm = 14.0

# Solution/electrolyte resistance control
# Values greater than 1 increase the solution resistance by reducing the
# electrolyte conductivity used by the SPMe/DFN electrolyte equations.
SOLUTION_RESISTANCE_MULTIPLIER = 1000.0
BASE_ELECTROLYTE_CONDUCTIVITY_S_M = 1.26

# Nyquist-loop shape controls
# Keep the SEI/charge-transfer branch and ohmic intercept, but avoid forcing
# the high-frequency branch to look like the real-axis return. The lower
# diffusivity/capacitance settings keep the visible real-axis approach tied to
# the low-frequency transport side of the spectrum instead.
#
# Increase these only if you deliberately want stronger overlap between the
# charge-transfer loop and the low-frequency transport curvature.
POSITIVE_DOUBLE_LAYER_CAPACITY_F_M2 = 2.5e-4
POSITIVE_PARTICLE_DIFFUSIVITY_M2_S = 8.0e-12

# Half-cell capacity estimate
Q_halfcell_Ah = 0.004

# Voltage limits for cathode vs Li/Li+
lower_cutoff_V = 3.0
upper_cutoff_V = 4.3

# Optional: use your measured cathode qOCV file
USE_CATHODE_QOCV_FROM_CSV = True

CATHODE_QOCV_FILE = Path(
    r"C:\Users\mugi_jo\Downloads\param\CG50 Prel-Parameter-Set 3C DLR"
    r"\N21700-BAK-CG50_electrode_level_data\OCV_measurements\cathode"
    r"\N21700-BAK-CG50-DLR-cathode-02-qOCV-lithiation.csv"
)

# Keep active SEI enabled so the SEI film resistance is maintained.
# Use the reduced mesh below if the active SEI solve becomes slow.
USE_ACTIVE_SEI = True

# Save output
save_results = True
output_csv = f"{MODEL_NAME}_CG50_cathode_vs_Li_halfcell_EIS_corrected.csv"


# =============================================================================
# FALLBACK OCP
# =============================================================================

from pybamm.input.parameters.lithium_ion.Chen2020 import nmc_LGM50_ocp_Chen2020


def lithium_metal_ocp(sto):
    """
    Li metal reference/counter electrode OCP vs Li/Li+.
    """
    return 0.0 * sto


# =============================================================================
# EXCHANGE-CURRENT DENSITY FUNCTIONS FROM CG50 PARAMETER SET
# =============================================================================

def PE_exch_current_density(c_e, c_s_surf, c_s_max, T):
    """
    Positive electrode / cathode exchange-current density.

    Larger value -> smaller charge-transfer resistance.
    Smaller value -> larger charge-transfer arc.
    """
    m_ref = 7.246393655878818e-8
    E_r = 17800.0

    arrhenius = np.exp(
        E_r / pybamm.constants.R * (1.0 / 298.15 - 1.0 / T)
    )

    return (
        m_ref
        * arrhenius
        * c_e**0.5
        * c_s_surf**0.5
        * (c_s_max - c_s_surf) ** 0.5
    )


def NE_exch_current_density(c_e, c_s_surf, c_s_max, T):
    """
    Negative electrode exchange-current density from the original full-cell set.

    It is mostly retained for compatibility. In positive-working-electrode
    half-cell mode, the negative branch is Li metal.
    """
    m_ref = 2.0645643855250177e-06
    E_r = 35000.0

    arrhenius = np.exp(
        E_r / pybamm.constants.R * (1.0 / 298.15 - 1.0 / T)
    )

    return (
        m_ref
        * arrhenius
        * c_e**0.5
        * c_s_surf**0.5
        * (c_s_max - c_s_surf) ** 0.5
    )


def lithium_metal_exchange_current_density(c_e, c_Li, T):
    """
    Li-metal exchange-current density.

    Increase this if you want the Li-metal counter/reference electrode to be
    almost non-limiting.
    """
    j0_ref = 1.0e1  # A.m-2
    return j0_ref * pybamm.ones_like(c_e)


# =============================================================================
# CATHODE qOCV READER
# =============================================================================

def read_cathode_qocv_lithiation_csv(file_path):
    """
    Reads cathode qOCV lithiation CSV and converts it into a PyBaMM OCP tuple.

    Expected columns:
        time (h), voltage (V), ah (Ah)
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(file_path)

    fieldnames = ["time (h)", "voltage (V)", "ah (Ah)"]
    data = {field: [] for field in fieldnames}

    with open(file_path, newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            for field in fieldnames:
                value = row[field].replace(",", ".")
                data[field].append(float(value))

    PE_lith_Cap = np.asarray(data["ah (Ah)"], dtype=float)
    PE_voltage = np.asarray(data["voltage (V)"], dtype=float)

    PE_lith_Cap_shift = PE_lith_Cap - PE_lith_Cap[-1]
    PE_Cap = PE_lith_Cap_shift / np.max(PE_lith_Cap_shift)
    PE_discharge_Cap = -(PE_Cap - PE_Cap[0])

    sto = np.asarray(PE_discharge_Cap, dtype=float)
    voltage = np.asarray(PE_voltage, dtype=float)

    # Downsample to avoid overly dense interpolant
    sto = sto[::20]
    voltage = voltage[::20]

    # Remove invalid points
    valid = np.isfinite(sto) & np.isfinite(voltage)
    sto = sto[valid]
    voltage = voltage[valid]

    # Sort and remove duplicate sto values
    order = np.argsort(sto)
    sto = sto[order]
    voltage = voltage[order]

    sto_unique, idx_unique = np.unique(sto, return_index=True)
    voltage_unique = voltage[idx_unique]

    # Keep valid stoichiometry range
    mask = (sto_unique >= 0.0) & (sto_unique <= 1.0)
    sto_unique = sto_unique[mask]
    voltage_unique = voltage_unique[mask]

    if len(sto_unique) < 5:
        raise ValueError("Too few valid cathode qOCV points.")

    ocp_tuple = ("OCV-PE_CG50", ([sto_unique], voltage_unique))

    return ocp_tuple, sto_unique, voltage_unique


def get_positive_ocp():
    """
    Returns cathode OCP function or PyBaMM interpolant tuple.
    """
    if USE_CATHODE_QOCV_FROM_CSV:
        try:
            ocp_tuple, sto, voltage = read_cathode_qocv_lithiation_csv(
                CATHODE_QOCV_FILE
            )
            print(f"Using cathode qOCV file:\n  {CATHODE_QOCV_FILE}")
            return ocp_tuple, sto, voltage

        except Exception as err:
            warnings.warn(
                "Could not read cathode qOCV file. "
                "Falling back to Chen2020 NMC OCP.\n"
                f"Original error: {err}",
                RuntimeWarning,
            )

    sto = np.linspace(0.001, 0.999, 500)
    voltage = nmc_LGM50_ocp_Chen2020(sto)

    print("Using fallback Chen2020 NMC OCP.")
    return nmc_LGM50_ocp_Chen2020, sto, voltage


# =============================================================================
# BUILD CG50 PARAMETER SET FOR CATHODE vs Li HALF-CELL
# =============================================================================

def make_CG50_cathode_vs_Li_parameter_values():
    """
    CG50 cathode vs Li metal half-cell parameter set.

    Positive branch:
        CG50 cathode / NMC

    Negative branch:
        Li metal counter/reference
    """

    positive_ocp, ocp_sto, ocp_voltage = get_positive_ocp()

    # -------------------------------------------------------------------------
    # Geometry
    # -------------------------------------------------------------------------
    if USE_EL_CELL_DISC_GEOMETRY:
        disc_radius_m = 0.5 * disc_diameter_mm * 1.0e-3
        A_cell = np.pi * disc_radius_m**2
        equivalent_side_m = np.sqrt(A_cell)

        electrode_height = equivalent_side_m
        electrode_width = equivalent_side_m

        L_n = 70.0e-6
        L_s = 1.07e-5
        L_p = 64.0e-6
        L_cc_n = 1.1e-5
        L_cc_p = 1.6e-5

        cell_volume = A_cell * (L_n + L_s + L_p + L_cc_n + L_cc_p)
        cooling_area = 2.0 * A_cell

        capacity_Ah = Q_halfcell_Ah
        current_A = Q_halfcell_Ah

    else:
        electrode_height = 0.3538361202590827
        electrode_width = 0.3538361202590827
        cooling_area = 0.00531
        cell_volume = 1.289286439267887e-05

        L_n = 70.0e-6
        L_s = 1.07e-5
        L_p = 64.0e-6
        L_cc_n = 1.1e-5
        L_cc_p = 1.6e-5

        capacity_Ah = 0.004
        current_A = 5.0

    parameter_values = pybamm.ParameterValues(
        {
            "chemistry": "lithium_ion",

            # -----------------------------------------------------------------
            # SEI parameters retained for optional later use
            # -----------------------------------------------------------------
            "Ratio of lithium moles to SEI moles": 2.0,
            "SEI partial molar volume [m3.mol-1]": 9.585e-05,
            "SEI reaction exchange current density [A.m-2]": 1.5e-10,
            "SEI resistivity [Ohm.m]": 1.6e6,
            "SEI solvent diffusivity [m2.s-1]": 2.5e-22,
            "Bulk solvent concentration [mol.m-3]": 2636.0,
            "SEI open-circuit potential [V]": 0.4,
            "SEI electron conductivity [S.m-1]": 8.95e-14,
            "SEI lithium interstitial diffusivity [m2.s-1]": 1.0e-20,
            "Lithium interstitial reference concentration [mol.m-3]": 15.0,
            "Initial SEI thickness [m]": 5.0e-09,
            "EC initial concentration in electrolyte [mol.m-3]": 4541.0,
            "EC diffusivity [m2.s-1]": 2.0e-18,
            "SEI kinetic rate constant [m.s-1]": 1.0e-12,
            "SEI growth activation energy [J.mol-1]": 0.0,

            # -----------------------------------------------------------------
            # Cell geometry
            # -----------------------------------------------------------------
            "Negative current collector thickness [m]": L_cc_n,
            "Negative electrode thickness [m]": L_n,
            "Separator thickness [m]": L_s,
            "Positive electrode thickness [m]": L_p,
            "Positive current collector thickness [m]": L_cc_p,

            "Electrode height [m]": electrode_height,
            "Electrode width [m]": electrode_width,
            "Cell cooling surface area [m2]": cooling_area,
            "Cell volume [m3]": cell_volume,
            "Cell thermal expansion coefficient [m.K-1]": 1.1e-06,

            # -----------------------------------------------------------------
            # Current collectors
            # -----------------------------------------------------------------
            "Negative current collector conductivity [S.m-1]": 58411000.0,
            "Positive current collector conductivity [S.m-1]": 36914000.0,
            "Negative current collector density [kg.m-3]": 8960.0,
            "Positive current collector density [kg.m-3]": 2700.0,
            "Negative current collector specific heat capacity [J.kg-1.K-1]": 385.0,
            "Positive current collector specific heat capacity [J.kg-1.K-1]": 897.0,
            "Negative current collector thermal conductivity [W.m-1.K-1]": 401.0,
            "Positive current collector thermal conductivity [W.m-1.K-1]": 237.0,

            # -----------------------------------------------------------------
            # Capacity and current
            # -----------------------------------------------------------------
            "Nominal cell capacity [A.h]": capacity_Ah,
            "Current function [A]": current_A,

            # Contact resistance shifts high-frequency intercept if enabled
            # "Contact resistance [Ohm]": 0.000164012,
            "Contact resistance [Ohm]": 5,
            # -----------------------------------------------------------------
            # Negative branch = Li metal
            # -----------------------------------------------------------------
            "Negative electrode OCP [V]": lithium_metal_ocp,

            # Retained for compatibility
            "Negative electrode double-layer capacity [F.m-2]": 0.00082,
            "Negative electrode exchange-current density [A.m-2]": NE_exch_current_density,
            "Negative electrode conductivity [S.m-1]": 236.0,
            "Negative particle diffusivity [m2.s-1]": 4.956824029646294e-13,
            "Negative electrode Bruggeman coefficient (electrolyte)": 0.4557018027399873,
            "Negative electrode Bruggeman coefficient (electrode)": 0.5,

            # Li metal specific parameters required by half-cell models
            "Lithium metal partial molar volume [m3.mol-1]": 1.3e-05,
            "Exchange-current density for lithium metal electrode [A.m-2]": lithium_metal_exchange_current_density,

            # -----------------------------------------------------------------
            # Positive branch = CG50 cathode
            # -----------------------------------------------------------------
            "Maximum concentration in positive electrode [mol.m-3]": 50000.0,
            "Positive electrode OCP [V]": positive_ocp,

            "Positive electrode porosity": 0.21,
            "Positive electrode active material volume fraction": 0.5329636099493749,
            "Positive particle radius [m]": 5.1e-06,
            "Positive electrode charge transfer coefficient": 0.5,

            # Larger capacitance shifts the interface loop to lower frequency,
            # keeping the second-loop minimum from landing on the real axis.
            "Positive electrode double-layer capacity [F.m-2]": (
                POSITIVE_DOUBLE_LAYER_CAPACITY_F_M2
            ),

            # Smaller exchange-current density increases Rct
            "Positive electrode exchange-current density [A.m-2]": PE_exch_current_density,
            # "Positive electrode exchange-current density [A.m-2]": 0.1,
            "Positive electrode density [kg.m-3]": 3262.0,
            "Positive electrode specific heat capacity [J.kg-1.K-1]": 700.0,
            "Positive electrode thermal conductivity [W.m-1.K-1]": 2.1,
            "Positive electrode OCP entropic change [V.K-1]": 0.0,

            # This is quite low and may strongly affect impedance.
            "Positive electrode conductivity [S.m-1]": .27240122531635543,

            # Faster solid diffusion brings the transport curvature into the
            # second-loop frequency window before that loop reaches the real axis.
            "Positive particle diffusivity [m2.s-1]": (
                POSITIVE_PARTICLE_DIFFUSIVITY_M2_S
            ),
            "Positive electrode Bruggeman coefficient (electrolyte)": 1.566481785533038,
            "Positive electrode Bruggeman coefficient (electrode)": 1.07,

            # -----------------------------------------------------------------
            # Separator
            # -----------------------------------------------------------------
            "Separator porosity": 0.31,
            "Separator density [kg.m-3]": 397.0,
            "Separator specific heat capacity [J.kg-1.K-1]": 700.0,
            "Separator thermal conductivity [W.m-1.K-1]": 0.16,
            "Separator Bruggeman coefficient (electrolyte)": 2.053759002360426,

            # -----------------------------------------------------------------
            # Electrolyte
            # -----------------------------------------------------------------
            # Lower electrolyte conductivity increases the electrolyte/solution
            # ohmic resistance and shifts the high-frequency Nyquist intercept
            # to the right.
            "Initial concentration in electrolyte [mol.m-3]": 1000.0,
            "Thermodynamic factor": 3.1,
            "Electrolyte diffusivity [m2.s-1]": 1.1e-12,
            "Electrolyte conductivity [S.m-1]": (
                BASE_ELECTROLYTE_CONDUCTIVITY_S_M / SOLUTION_RESISTANCE_MULTIPLIER
            ),
            "Cation transference number": 0.142,

            # -----------------------------------------------------------------
            # Temperature and thermal
            # -----------------------------------------------------------------
            "Reference temperature [K]": 298.15,
            "Ambient temperature [K]": 298.15,
            "Initial temperature [K]": 298.15,
            "Total heat transfer coefficient [W.m-2.K-1]": 10.0,

            # -----------------------------------------------------------------
            # Voltage limits
            # -----------------------------------------------------------------
            "Lower voltage cut-off [V]": lower_cutoff_V,
            "Upper voltage cut-off [V]": upper_cutoff_V,
            "Open-circuit voltage at 0% SOC [V]": lower_cutoff_V,
            "Open-circuit voltage at 100% SOC [V]": upper_cutoff_V,

            # -----------------------------------------------------------------
            # Default initial concentrations.
            # These will be overwritten for each SOC by make_parameter_values_for_soc().
            # -----------------------------------------------------------------
            "Initial concentration in negative electrode [mol.m-3]": 0.98062035 * 31695.0,
            "Initial concentration in positive electrode [mol.m-3]": 3.58369851e-04*50000,

            # -----------------------------------------------------------------
            # Other PyBaMM bookkeeping
            # -----------------------------------------------------------------
            "Number of electrodes connected in parallel to make a cell": 1.0,
            "Number of cells connected in series to make a battery": 1.0,

            "Negative electrode reaction-driven LAM factor [m3.mol-1]": 0.0,
            "Positive electrode reaction-driven LAM factor [m3.mol-1]": 0.0,

            "Exchange current density reference value in negative electrode": 2.0645643855250177e-06,
            "Exchange current density reference value in positive electrode": 7.246393655878818e-05,
        }
    )

    return parameter_values, ocp_sto, ocp_voltage


parameter_values, ocp_sto, ocp_voltage = make_CG50_cathode_vs_Li_parameter_values()


# =============================================================================
# MODEL OPTIONS
# =============================================================================

model_options = {
    "surface form": "differential",
    "working electrode": "positive",
    "contact resistance": "true",
}

if USE_ACTIVE_SEI:
    # model_options.update(
    #     {
    #         "SEI": "ec reaction limited",
    #         "SEI film resistance": "distributed",
    #         "SEI porosity change": "false",
    #     }
    # )
    model_options.update({
        "surface form": "differential",
        "working electrode": "positive",
        "contact resistance": "true",
        "SEI": "ec reaction limited",
        "SEI film resistance": "distributed",
        "SEI porosity change": "false",
    })
# Reduced mesh for stability. Increase after confirming the model behaves correctly.
var_pts = {"x_n": 12, "x_s": 12, "x_p": 32, "r_n": 12, "r_p": 32}

def make_model(model_name):
    model_name = model_name.lower()

    if model_name == "spm":
        return pybamm.lithium_ion.SPM(options=model_options)

    if model_name == "spme":
        return pybamm.lithium_ion.SPMe(options=model_options)

    if model_name == "dfn":
        return pybamm.lithium_ion.DFN(options=model_options)

    raise ValueError("MODEL_NAME must be 'SPM', 'SPMe', or 'DFN'.")


# =============================================================================
# SOC / STOICHIOMETRY CONTROL
# =============================================================================

def positive_sto_from_soc(soc):
    """
    Convert user SOC value into positive-electrode stoichiometry.

    For a cathode vs Li half-cell, the measured coordinate may be either direct
    or reversed depending on how the qOCV file was processed.
    """
    soc = float(soc)

    if POSITIVE_STO_MAPPING.lower() == "direct":
        sto = soc
    elif POSITIVE_STO_MAPPING.lower() == "one_minus":
        sto = 1.0 - soc
    else:
        raise ValueError("POSITIVE_STO_MAPPING must be 'direct' or 'one_minus'.")

    return float(np.clip(sto, 1.0e-4, 0.9999))


def safe_parameter_update(parameter_values_i, updates):
    """
    Update ParameterValues across PyBaMM versions.
    """
    try:
        parameter_values_i.update(updates, check_already_exists=True)
    except TypeError:
        parameter_values_i.update(updates)

    return parameter_values_i


def make_parameter_values_for_soc(base_parameter_values, soc):
    """
    Make a fresh parameter set where positive electrode initial concentration
    is set directly from the desired positive stoichiometry.
    """
    parameter_values_i = base_parameter_values.copy()

    sto_p = positive_sto_from_soc(soc)
    c_p_max = parameter_values_i["Maximum concentration in positive electrode [mol.m-3]"]

    c_p_init = sto_p * c_p_max
    parameter_values_i = safe_parameter_update(
        parameter_values_i,
        {
            "Initial concentration in positive electrode [mol.m-3]": c_p_init,
        },
    )

    # Sanity check: this catches the bug where the SOC-to-stoichiometry mapping
    # is changed in the legend but not actually applied to PyBaMM's initial state.
    applied_c_p_init = float(
        parameter_values_i["Initial concentration in positive electrode [mol.m-3]"]
    )
    if not np.isclose(applied_c_p_init, c_p_init, rtol=0.0, atol=1.0e-10):
        raise RuntimeError(
            "Positive-electrode initial concentration was not updated for SOC "
            f"{soc}. Expected {c_p_init:.12e}, got {applied_c_p_init:.12e}."
        )

    return parameter_values_i, sto_p


# =============================================================================
# EIS HELPERS
# =============================================================================

def extract_eis_result(result):
    """
    Robust EIS extraction across PyBaMM versions.
    """
    try:
        Z_re = np.asarray(result["Z_re [Ohm]"], dtype=float)
        Z_im = np.asarray(result["Z_im [Ohm]"], dtype=float)
        Z = Z_re + 1j * Z_im
    except Exception:
        try:
            Z = np.asarray(result["Impedance [Ohm]"], dtype=complex)
        except Exception:
            Z = np.asarray(result["Impedance"], dtype=complex)

    try:
        f = np.asarray(result["Frequency [Hz]"], dtype=float)
    except Exception:
        f = frequencies

    return f, Z


def nyquist_y(Z):
    return -np.imag(Z)


def orient_for_nyquist(Z):
    """
    Orient so capacitive features appear positive in -Im(Z).
    """
    Z = np.asarray(Z, dtype=complex)

    if np.nanmedian(nyquist_y(Z)) < 0:
        Z = np.conjugate(Z)

    return Z


def dominant_time_constant(f, Z):
    y = nyquist_y(Z)
    idx = int(np.argmax(y))

    f_peak = float(f[idx])
    tau_peak = 1.0 / (2.0 * np.pi * f_peak)

    return f_peak, tau_peak


def subtract_high_frequency_intercept(f, Z):
    """
    Subtract high-frequency real-axis intercept for visual comparison.
    """
    hf_idx = int(np.argmax(f))
    R_hf = float(np.real(Z[hf_idx]))
    Z_corr = Z - R_hf

    return Z_corr, R_hf




def PE_exch_current_density_numeric(c_e, c_s_surf, c_s_max, T):
    """
    Numerical version of PE_exch_current_density for printing diagnostics.
    This matches the same formula used in the PyBaMM parameter function.
    """
    m_ref = 7.246393655878818e-8
    E_r = 17800.0
    R = 8.31446261815324  # J mol-1 K-1

    arrhenius = np.exp(E_r / R * (1.0 / 298.15 - 1.0 / T))

    return (
        m_ref
        * arrhenius
        * c_e**0.5
        * c_s_surf**0.5
        * (c_s_max - c_s_surf) ** 0.5
    )

####printing exchange current


def print_positive_exchange_current_density(parameter_values_i, sto_p):
    """
    Print the base-state positive electrode exchange-current density
    used at the selected positive electrode stoichiometry.
    """
    c_e = float(parameter_values_i["Initial concentration in electrolyte [mol.m-3]"])
    c_s_max = float(parameter_values_i["Maximum concentration in positive electrode [mol.m-3]"])
    T = float(parameter_values_i["Reference temperature [K]"])

    c_s_surf = sto_p * c_s_max

    j0_p = PE_exch_current_density_numeric(
        c_e=c_e,
        c_s_surf=c_s_surf,
        c_s_max=c_s_max,
        T=T,
    )

    print("\nPositive electrode exchange-current density diagnostic:")
    print(f"  c_e       = {c_e:.6e} mol.m-3")
    print(f"  c_s_surf  = {c_s_surf:.6e} mol.m-3")
    print(f"  c_s_max   = {c_s_max:.6e} mol.m-3")
    print(f"  sto_p     = {sto_p:.6f}")
    print(f"  T         = {T:.2f} K")
    print(f"  j0_p      = {j0_p:.6e} A.m-2")

    return j0_p

# =============================================================================
# PRINT BASIC DIAGNOSTICS
# =============================================================================

R_p = parameter_values["Positive particle radius [m]"]
D_p = parameter_values["Positive particle diffusivity [m2.s-1]"]

tau_s = R_p**2 / D_p
f_s = 1.0 / (2.0 * np.pi * tau_s)

print("\n================ MODEL DIAGNOSTICS ================")
print(f"MODEL_NAME: {MODEL_NAME}")
print(f"USE_ACTIVE_SEI: {USE_ACTIVE_SEI}")
print(f"Solution resistance multiplier: {SOLUTION_RESISTANCE_MULTIPLIER:.3g}x")
print(
    "Effective electrolyte conductivity: "
    f"{parameter_values['Electrolyte conductivity [S.m-1]']:.6g} S/m"
)
print(f"Frequency range: {frequencies.min():.3e} Hz to {frequencies.max():.3e} Hz")
print(f"Number of frequency points: {len(frequencies)}")
print(f"Positive particle radius: {R_p:.3e} m")
print(f"Positive particle diffusivity: {D_p:.3e} m2/s")
print(f"Estimated positive solid diffusion time scale: {tau_s:.3e} s")
print(f"Estimated positive solid diffusion frequency: {f_s:.3e} Hz")
print("If the lower frequency limit is much larger than this, the diffusion tail may not appear.")



# =============================================================================
# RUN EIS SIMULATION
# =============================================================================

all_results = {}

for soc in soc_list:
    print("\n" + "=" * 80)
    print(f"Simulating {MODEL_NAME} CG50 cathode vs Li half-cell")
    print(f"Nominal SOC input = {soc:.3f}")
    print("=" * 80)

    parameter_values_i, sto_p = make_parameter_values_for_soc(parameter_values, soc)
    j0_p = print_positive_exchange_current_density(parameter_values_i, sto_p)

    # Estimate OCP from qOCV table
    U_p_est = np.interp(sto_p, ocp_sto, ocp_voltage)

    print(f"Positive stoichiometry used = {sto_p:.6f}")
    print(f"Estimated cathode OCP       = {U_p_est:.6f} V vs Li/Li+")

    model_i = make_model(MODEL_NAME)

    try:
        eis_sim_i = pybamm.EISSimulation(
            model_i,
            parameter_values=parameter_values_i,
            var_pts=var_pts,
        )
    except TypeError:
        warnings.warn(
            "This PyBaMM version did not accept var_pts in EISSimulation. "
            "Using default discretisation.",
            RuntimeWarning,
        )

        eis_sim_i = pybamm.EISSimulation(
            model_i,
            parameter_values=parameter_values_i,
        )

    # Important:
    # Do not pass initial_soc here. We already set the positive electrode
    # initial concentration directly.
    # result = eis_sim_i.solve(frequencies,
    #     initial_soc=soc,)
    result = eis_sim_i.solve(frequencies)

    f_result, Z = extract_eis_result(result)
    Z = orient_for_nyquist(Z)

    Z_corr, R_hf = subtract_high_frequency_intercept(f_result, Z)
    f_peak, tau_peak = dominant_time_constant(f_result, Z)

    # all_results[soc] = {
    #     "frequency": f_result,
    #     "Z": Z,
    #     "Z_corr": Z_corr,
    #     "R_hf": R_hf,
    #     "positive_sto": sto_p,
    #     "estimated_ocp": U_p_est,
    #     "f_peak": f_peak,
    #     "tau_peak": tau_peak,
    # }

    all_results[soc] = {
        "frequency": f_result,
        "Z": Z,
        "Z_corr": Z_corr,
        "R_hf": R_hf,
        "positive_sto": sto_p,
        "estimated_ocp": U_p_est,
        "positive_j0_A_m2": j0_p,
        "f_peak": f_peak,
        "tau_peak": tau_peak,
    }
    print(f"High-frequency real intercept: {R_hf:.6e} Ohm")
    print(f"Peak -Im(Z): f = {f_peak:.6g} Hz")
    print(f"Approx. dominant time constant: tau = {tau_peak:.6g} s")


# =============================================================================
# PLOT 0: OCP CHECK
# =============================================================================

plt.figure(figsize=(6, 5))
plt.plot(ocp_sto, ocp_voltage, "o-", markersize=3)

for soc in soc_list:
    sto_p = all_results[soc]["positive_sto"]
    U_p_est = all_results[soc]["estimated_ocp"]
    plt.plot(sto_p, U_p_est, "s", markersize=7, label=f"SOC={soc:.2f}")

plt.xlabel("Positive electrode stoichiometry / SOC-like coordinate [-]")
plt.ylabel("Cathode OCP vs Li/Li+ [V]")
plt.title("Cathode OCP used in the CG50 half-cell simulation")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()


# =============================================================================
# PLOT 1: RAW NYQUIST PLOT
# =============================================================================

plt.figure(figsize=(6, 5))
colors = plt.get_cmap("tab10")

for ii, soc in enumerate(soc_list):
    Z = all_results[soc]["Z"]
    sto_p = all_results[soc]["positive_sto"]

    plt.plot(
        np.real(Z),
        -np.imag(Z),
        "o-",
        markersize=4,
        linewidth=1.8,
        color=colors(ii),
        label=f"SOC={soc:.2f}, sto_p={sto_p:.2f}",
    )

plt.xlabel(r"$Z_\mathrm{Re}$ [$\Omega$]")
plt.ylabel(r"$-Z_\mathrm{Im}$ [$\Omega$]")
plt.title(f"{MODEL_NAME} EIS: CG50 cathode vs Li metal half-cell")
plt.grid(True)
plt.legend(fontsize=8)

# Axis equal for Nyquist plot
plt.axis("equal")

plt.tight_layout()
plt.show()
# =============================================================================
# PLOT 2: HIGH-FREQUENCY-INTERCEPT-CORRECTED NYQUIST PLOT
# =============================================================================

plt.figure(figsize=(6, 5))

for ii, soc in enumerate(soc_list):
    Z_corr = all_results[soc]["Z_corr"]
    sto_p = all_results[soc]["positive_sto"]
    R_hf = all_results[soc]["R_hf"]

    plt.plot(
        np.real(Z_corr),
        -np.imag(Z_corr),
        "o-",
        markersize=4,
        linewidth=1.8,
        color=colors(ii),
        label=f"SOC={soc:.2f}, sto_p={sto_p:.2f}, R_hf={R_hf:.3g} Ω",
    )

plt.xlabel(r"$Z_\mathrm{Re} - R_\mathrm{hf}$ [$\Omega$]")
plt.ylabel(r"$-Z_\mathrm{Im}$ [$\Omega$]")
plt.title(f"{MODEL_NAME} EIS: high-frequency-intercept corrected")
plt.grid(True)
plt.legend(fontsize=8)
plt.axis("equal")
plt.tight_layout()
plt.show()


# =============================================================================
# SAVE RESULTS
# =============================================================================

if save_results:
    rows = []

    for soc in soc_list:
        result_i = all_results[soc]
        frequency_i = result_i["frequency"]
        Z_i = result_i["Z"]
        Z_corr_i = result_i["Z_corr"]

        for freq, z_val, z_corr_val in zip(frequency_i, Z_i, Z_corr_i):
            rows.append(
                {
                    "soc_input": soc,
                    "positive_stoichiometry": result_i["positive_sto"],
                    "estimated_ocp_V_vs_Li": result_i["estimated_ocp"],
                    "positive_j0_A_m2": result_i["positive_j0_A_m2"],
                    "R_hf_ohm": result_i["R_hf"],
                    "f_peak_Hz": result_i["f_peak"],
                    "tau_peak_s": result_i["tau_peak"],
                    "frequency_Hz": freq,
                    "Z_real_ohm": np.real(z_val),
                    "Z_imag_ohm": np.imag(z_val),
                    "minus_Z_imag_ohm": -np.imag(z_val),
                    "Z_corr_real_ohm": np.real(z_corr_val),
                    "Z_corr_imag_ohm": np.imag(z_corr_val),
                    "minus_Z_corr_imag_ohm": -np.imag(z_corr_val),
                }
            )

    pd.DataFrame(rows).to_csv(output_csv, index=False)
    print(f"Saved EIS results to {output_csv}")


